# Module 09 — Kafka / Redpanda (bus d'événements)

Découpler `poll` et `parse` via un topic Kafka compatible.

**Prérequis** :
```bash
docker compose up -d redpanda
# .env : KAFKA_BOOTSTRAP_SERVERS=localhost:19092
```

Tuto : [`docs/modules/09-kafka-redpanda.md`](../docs/modules/09-kafka-redpanda.md)

## Étape 1 — Vérifier la config Kafka

Si `enabled` est `False`, ajoute `KAFKA_BOOTSTRAP_SERVERS=localhost:19092` dans `.env`.

In [ ]:
from presslake.events.config import CONSUMER_GROUP_PARSE, bootstrap_servers
from presslake.events.producer import EventProducer
from presslake.events.topics import TOPIC_ARTICLES_INGESTED

servers = bootstrap_servers()
print("bootstrap:", servers)
print("topic:", TOPIC_ARTICLES_INGESTED)
print("consumer group:", CONSUMER_GROUP_PARSE)

producer = EventProducer()
print("producer enabled:", producer.enabled)
producer.close()

## Étape 2 — Lire les événements du topic

On lit au plus 5 messages (timeout 2 s si le topic est vide ou à jour).

In [ ]:
from presslake.events.consumer import iter_article_ingested

events = list(iter_article_ingested(replay=True, limit=5))
print(f"{len(events)} événement(s) lu(s) (replay, max 5)")
for ev in events:
    print(f"- {ev['feed_id']} | {ev['url'][:70]}…")

## Étape 3 — Publier un événement test (Python)

Même format que `poll` après un nouvel article. **Attention** : utilise un `s3_uri` réel si tu veux parser ensuite.

In [ ]:
from presslake.events.producer import EventProducer

with EventProducer() as producer:
    if not producer.enabled:
        raise RuntimeError("KAFKA_BOOTSTRAP_SERVERS manquant")
    producer.publish_article_ingested(
        feed_id="notebook-test",
        url="https://example.com/notebook-kafka-test",
        s3_uri="s3://presslake/bronze/source=notebook-test/dt=2026-08-31/" + "c" * 64 + ".json",
        content_hash="c" * 64,
        item_key="notebook-kafka-test",
    )
print("publié sur presslake.articles.ingested")

## Étape 4 — Parser via le bus

Équivalent CLI :
```bash
uv run presslake parse --from-kafka --limit 3
uv run presslake parse --from-kafka --replay --limit 3
```

`--replay` = offset 0 + `reparse=True` (reconstruit le silver même si déjà parsed).

In [ ]:
from presslake.parse.run import parse_from_kafka

# Décommente une ligne pour tester (peut prendre quelques secondes)
# parse_from_kafka(limit=2)
# parse_from_kafka(replay=True, limit=2)
print("Décommente parse_from_kafka(...) pour lancer le parse via Kafka")

## Étape 5 — Chaîne bout en bout (terminal)

```bash
# Terminal 1 : écouter
docker compose exec redpanda rpk topic consume presslake.articles.ingested -o end

# Terminal 2 : ingest + parse découplé
uv run presslake poll
uv run presslake parse --from-kafka
```

### Schéma

```
poll → bronze + Postgres + Kafka
                    ↓
         presslake.articles.ingested
                    ↓
         parse --from-kafka → silver
```